# 06 — Ablation Study

**Critical for grade:** Quantify the contribution of NLP features to model performance.

We compare:
1. **Stats-only** — baseline using FIFA/TransferMarkt features alone
2. **Stats + NLP** — adding sentiment, hype, injury score, etc.

Expected: NLP features improve accuracy and F1-macro.

In [ ]:
import sys
sys.path.insert(0, '../models/ml_classification')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score, accuracy_score
from pathlib import Path

SAVED = Path('../models/ml_classification/saved_models')
LABEL_ORDER = ['Average', 'Good', 'Excellent', 'World Class']

## 1. Load Models & Test Splits

In [ ]:
def load_and_eval(model_name, suffix):
    model_path = SAVED / f'{model_name}_{suffix}.pkl'
    test_path = SAVED / f'test_split_{suffix}.csv'
    if not model_path.exists() or not test_path.exists():
        print(f'  Missing: {model_name} / {suffix}')
        return None
    bundle = joblib.load(model_path)
    df_test = pd.read_csv(test_path)
    y_test = df_test.pop('label').values
    features = bundle['features']
    X_test = df_test[features].fillna(df_test[features].median(numeric_only=True))
    y_pred = bundle['model'].predict(X_test)
    return {
        'model': model_name,
        'suffix': suffix,
        'accuracy': accuracy_score(y_test, y_pred),
        'f1_macro': f1_score(y_test, y_pred, average='macro'),
        'y_test': y_test,
        'y_pred': y_pred,
    }

models = ['logistic_regression', 'random_forest', 'xgboost']
results = []
for m in models:
    for s in ['no_nlp', 'with_nlp']:
        r = load_and_eval(m, s)
        if r:
            results.append(r)

df_res = pd.DataFrame([{k:v for k,v in r.items() if k not in ('y_test','y_pred')} for r in results])
df_res

## 2. Delta Table (NLP Impact)

In [ ]:
pivot_acc = df_res.pivot(index='model', columns='suffix', values='accuracy')
pivot_f1  = df_res.pivot(index='model', columns='suffix', values='f1_macro')

pivot_acc['delta_acc'] = pivot_acc['with_nlp'] - pivot_acc['no_nlp']
pivot_f1['delta_f1']   = pivot_f1['with_nlp']  - pivot_f1['no_nlp']

print('=== Accuracy ===' )
print(pivot_acc.to_string())
print()
print('=== F1-Macro ===')
print(pivot_f1.to_string())

## 3. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, pivot, title in [
    (axes[0], pivot_acc[['no_nlp','with_nlp']], 'Accuracy'),
    (axes[1], pivot_f1[['no_nlp','with_nlp']],  'F1-Macro'),
]:
    pivot.plot(kind='bar', ax=ax, color=['#ef9a9a','#a5d6a7'], edgecolor='white')
    ax.set_title(f'{title}: Stats-Only vs. Stats+NLP')
    ax.set_ylabel(title)
    ax.set_ylim(0, 1)
    ax.legend(['No NLP', 'With NLP'])
    ax.tick_params(axis='x', rotation=20)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}', 
                    (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.suptitle('Ablation Study: NLP Feature Contribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../models/ml_classification/saved_models/ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Conclusions

Fill in after running:

| Model | No NLP Acc | +NLP Acc | Delta |
|-------|-----------|----------|-------|
| LR    | TBD       | TBD      | TBD   |
| RF    | TBD       | TBD      | TBD   |
| XGB   | TBD       | TBD      | **+X%** |

**Finding:** NLP features improved XGBoost accuracy by X percentage points,
demonstrating that media sentiment is a meaningful signal for performance prediction.